In [1]:
import sys,os
import traceback
sys.path.append(r'C:/data/EnergyTrading/Python/')

from Database.DB_writer import db_writer
from Database.DB_reader import Database


In [2]:
from Utilities.email_sending import send_plain_email, send_html_email
EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD') # the password needs to be set as EMAIL_PASSWORD in system variables of the computer where the process is running, it is located in S:/Algo/email_password.txt
if EMAIL_PASSWORD is None:
    raise ValueError("EMAIL_PASSWORD environment variable not set")

RECIPIENT = "scasny_martin@energytrading.sk" # can also be a list of recipients

In [4]:
if __name__ == "__main__":
    db_w = db_writer()
    db_r = Database()

    countries = ['at', 'be', 'cz', 'de', 'dkw', 'dke',
                 'fr','hu','nl','sk', 'si', 'ro', 'bg']
    countries = ['cz', 'hu', 'ro']
    result = {c: dict(country=c, message='No data') for c in countries}
    try:
        for country in countries:
            if db_w.spot_write(country) == -1:
                db_r.merge_from_staging_to_prod_enum(schema='spot', table=country)
                result[country]['message'] = "Spot prices contains NULL value(s)."
                continue
            
            db_r.merge_from_staging_to_prod_enum(schema='spot', table=country)
            result[country]['message'] = "Success"

        # Capture the traceback
        error_traceback = traceback.format_exc()
        # Format dictionary items for HTML content
        items_html = '\n'.join([f'<p>{key}: {value}</p>' for key, value in result.items()])
        html_content = f"""\
        <html>
            <body>
                <h1>Data updated</h1>
                <p>{items_html}</p>
            </body>
        </html>
        """

        send_html_email(
            RECIPIENT, 
            "AUTOMATIC JOBS - REPORT - spot_daily_update", 
            "", 
            html_content, 
            "",
            email_password=EMAIL_PASSWORD
        )
    except Exception as e:
        error_message = str(e)

        # Capture the traceback
        error_traceback = traceback.format_exc()
        html_content = f"""\
        <html>
            <body>
                <h1>Failed job spot_daily_update</h1>
                <p>{error_message}</p>
                <pre>{error_traceback}</pre>
            </body>
        </html>
        """

        send_html_email(
            RECIPIENT, 
            "AUTOMATIC JOBS - FAILED - spot_daily_update", 
            "", 
            html_content, 
            "",
            email_password=EMAIL_PASSWORD
        )
    

Scrape result for CZ is not a dataframe
Connected to the database postgre


C:\Users\scasny\AppData\Local\Temp\ipykernel_32028\4243756052.py:11: RuntimeWarning: coroutine 'submit_autonomous_task_run_to_engine' was never awaited
  if db_w.spot_write(country) == -1:


KeyboardInterrupt: 